In [7]:
# Import Libraries
import pandas as pd
import numpy as np
import pickle
import json

### Load Model & File Pendukung

In [8]:
with open('../src/model_rf_tuned.pkl', 'rb') as file_1:
    model = pickle.load(file_1)

with open('../src/list_top_categories.txt', 'r') as file_2:
    list_top_categories = json.load(file_2)

print(list_top_categories)

{'OperatingSystems': ['2', '1', '3', '4'], 'TrafficType': ['2', '1', '3', '4', '13', '10', '6', '8', '5', '11', '20']}


### Data Inference
* Data dibuat manual dengan **format yang sama seperti dataset asli** (17 kolom fitur, belum diproses). Data ini **tidak termasuk** train-set maupun test-set.
* Ada 3 sesi dengan perilaku berbeda:
    * **Sesi 1:** pengunjung baru, membuka banyak halaman produk dan mengunjungi halaman bernilai (bulan November).
    * **Sesi 2:** pengunjung lama, hanya membuka sedikit halaman produk tanpa halaman bernilai (bulan Mei).
    * **Sesi 3:** pengunjung lama dengan aktivitas sedang, datang dari sumber traffic yang jarang muncul (bulan Juni).

In [9]:
# Data inference dalam format asli (tipe data sama seperti dataset asli)
data_inf = pd.DataFrame({
    'Administrative': [4, 0, 2],
    'Administrative_Duration': [120.5, 0.0, 45.0],
    'Informational': [1, 0, 0],
    'Informational_Duration': [30.0, 0.0, 0.0],
    'ProductRelated': [45, 12, 25],
    'ProductRelated_Duration': [2100.0, 380.0, 900.0],
    'BounceRates': [0.005, 0.02, 0.01],
    'ExitRates': [0.015, 0.045, 0.025],
    'PageValues': [35.5, 0.0, 0.0],
    'SpecialDay': [0.0, 0.6, 0.0],
    'Month': ['Nov', 'May', 'June'],
    'OperatingSystems': [2, 3, 1],
    'Browser': [2, 1, 4],
    'Region': [1, 3, 2],
    'TrafficType': [2, 13, 7],
    'VisitorType': ['New_Visitor', 'Returning_Visitor', 'Returning_Visitor'],
    'Weekend': [False, True, False]
}, index=['Sesi 1', 'Sesi 2', 'Sesi 3'])

data_inf

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
Sesi 1,4,120.5,1,30.0,45,2100.0,0.005,0.015,35.5,0.0,Nov,2,2,1,2,New_Visitor,False
Sesi 2,0,0.0,0,0.0,12,380.0,0.020,0.045,0.0,0.6,May,3,1,3,13,Returning_Visitor,True
Sesi 3,2,45.0,0,0.0,25,900.0,0.010,0.025,0.0,0.0,June,1,4,2,7,Returning_Visitor,False


### Penyesuaian Data Inference
Data diproses sama seperti saat training:
* **Format data:** `June` → `Jun`, kolom kode → string, `Weekend` → 0/1.
* **Cardinality:** kategori `OperatingSystems` dan `TrafficType` yang tidak ada di train-set diubah menjadi `'Other'`.
* **Drop kolom:** `Region`, `Browser`, dan `BounceRates`.
* Capping, scaling, dan encoding dilakukan otomatis oleh pipeline.

In [10]:
# 1. Penyesuaian data seperti pada Data Loading
data_inf['Month'] = data_inf['Month'].replace('June', 'Jun')
code_cols = ['OperatingSystems', 'Browser', 'Region', 'TrafficType']
data_inf[code_cols] = data_inf[code_cols].astype(str)
data_inf['Weekend'] = data_inf['Weekend'].astype(int)

# 2. Handling cardinality, kategori di luar daftar train-set digabung menjadi 'Other'
for col in ['OperatingSystems', 'TrafficType']:
    data_inf[col] = np.where(data_inf[col].isin(list_top_categories[col]), data_inf[col], 'Other')

# 3. Drop kolom hasil feature selection agar kolom sama dengan saat training
data_inf_selected = data_inf.drop(['Region', 'Browser', 'BounceRates'], axis=1)

data_inf_selected

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,TrafficType,VisitorType,Weekend
Sesi 1,4,120.5,1,30.0,45,2100.0,0.015,35.5,0.0,Nov,2,2,New_Visitor,0
Sesi 2,0,0.0,0,0.0,12,380.0,0.045,0.0,0.6,May,3,13,Returning_Visitor,1
Sesi 3,2,45.0,0,0.0,25,900.0,0.025,0.0,0.0,Jun,1,Other,Returning_Visitor,0


### Prediksi

In [11]:
# Prediksi kelas & probabilitas membeli
hasil = pd.DataFrame(index=data_inf_selected.index)
hasil['prediksi'] = model.predict(data_inf_selected)
hasil['probabilitas_beli'] = model.predict_proba(data_inf_selected)[:, 1].round(3)
hasil['keterangan'] = hasil['prediksi'].map({1: 'Berpotensi membeli', 0: 'Tidak berpotensi membeli'})

hasil

,prediksi,probabilitas_beli,keterangan
Sesi 1,1,0.873,Berpotensi membeli
Sesi 2,0,0.004,Tidak berpotensi membeli
Sesi 3,0,0.181,Tidak berpotensi membeli


### Kesimpulan Inference
| Sesi | Prediksi | Probabilitas | Alasan Utama | Rekomendasi |
|---|---|---|---|---|
| 1 | Membeli | 0.873 | Banyak lihat produk & mengunjungi halaman bernilai (`PageValues` 35.5) | Tidak perlu promo, cukup pastikan checkout lancar |
| 2 | Tidak membeli | 0.004 | Sedikit lihat produk & tanpa halaman bernilai | Bukan prioritas promo |
| 3 | Tidak membeli | 0.181 | Aktivitas sedang, tapi tanpa halaman bernilai | Cukup email pengingat produk |

* Hasil prediksi sesuai dengan temuan EDA: **`PageValues` adalah faktor paling menentukan** apakah pengunjung membeli.